## 08 — Coverage audit, pipeline re-run & tiles GPKG repair

Two distinct coverage gaps are addressed here:

**Gap 1 — Unprocessed cities** (inputs present, sentinel missing):
Cities with candidate parquets in `data/01_raw/{city}/vector/` but no
`vector_metrics_tiles_all_datasets.parquet` in `outputs/metrics/{city}/`.
These need a pipeline re-run.

**Gap 2 — Missing tiles GPKG** (sentinel present, tiles GPKG missing):
Cities where the pipeline completed (sentinel exists) but the tile geometry
file `data/01_raw/{city}/tiles/{city}_tiles.gpkg` was cleaned from Drive
afterwards. Notebook 07 cannot enrich these cities without the tile geometries.
These need tiles GPKG regeneration from the AOI, not a full re-run.

**Steps**
1. Read tracker + scan disk → build city × dataset coverage table (with `Suitable` flag)
2. Gap 1: flag and re-run unprocessed cities (suitable + AOI on disk only)
3. Gap 2: regenerate tiles GPKG for complete cities missing the file
4. Verify results
5. Regenerate `vector_all_cities_merged.xlsx` from all per-city summaries

In [ ]:
!pip install -q openpyxl

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, tempfile
from pathlib import Path
import pandas as pd
import yaml

CONFIG_PATH  = Path('/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/configs/validation_configs.yaml')
PROJECT_ROOT = CONFIG_PATH.parents[1]
# Code from GitHub, data from Drive (see colab_bootstrap.py in the repo).
# Drive PROJECT_ROOT/src is a stale hand-copy; never import from it.
import subprocess as _sp
_sp.run(['wget','-q','-O','/content/colab_bootstrap.py','https://raw.githubusercontent.com/GFDRR/urban_validation/fix/pipeline-audit/colab_bootstrap.py'], check=False)
sys.path.insert(0, '/content')
from colab_bootstrap import setup as _setup
_setup(PROJECT_ROOT)

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
cfg['root_dir'] = str(PROJECT_ROOT)

DATA_DIR     = PROJECT_ROOT / cfg['data_dir']          # data/01_raw/
METRICS_ROOT = PROJECT_ROOT / 'outputs' / 'metrics'
GLOBAL_OUT   = PROJECT_ROOT / 'outputs' / 'global_metrics'
GLOBAL_OUT.mkdir(parents=True, exist_ok=True)

SENTINEL     = 'vector_metrics_tiles_all_datasets.parquet'
CITY_SUMMARY = 'vector_city_summary_all_datasets.parquet'
CAND_NAMES   = [
    d['name']
    for d in cfg.get('vector', {}).get('datasets', [])
    if d.get('enabled', True)
]

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'DATA_DIR     : {DATA_DIR}  exists={DATA_DIR.exists()}')
print(f'METRICS_ROOT : {METRICS_ROOT}  exists={METRICS_ROOT.exists()}')
print(f'Candidate datasets : {CAND_NAMES}')

In [ ]:
# ── Audit scan ───────────────────────────────────────────────────────────────
# Universe = all city slugs seen in DATA_DIR or METRICS_ROOT.
# Suitable status is read from the tracker CSV — same filter the pipeline applies.

# Read tracker for Suitable status
tracker_path = PROJECT_ROOT / cfg['aoi_tracker']
tracker_df = pd.read_csv(tracker_path, dtype=str)
tracker_df.columns = tracker_df.columns.str.strip()
tracker_df = tracker_df.apply(lambda c: c.str.strip() if c.dtype == object else c)

suitable_col = next((c for c in tracker_df.columns if 'suitable' in c.lower()), None)
suitable_map: dict = {}
if suitable_col:
    for _, row in tracker_df.iterrows():
        city = str(row.get('dataset_folder_name', '') or '').strip().lower()
        if city:
            suitable_map[city] = str(row.get(suitable_col, '')).lower() == 'yes'
    print(f'Tracker: {len(suitable_map)} cities | suitable col: {suitable_col!r}')
    print(f'  Suitable=yes: {sum(suitable_map.values())} | Suitable=no/missing: {sum(not v for v in suitable_map.values())}')
else:
    print('[WARN] No "Suitable" column found in tracker — suitable flag will be None for all cities.')

# Discover cities from disk
data_cities = set()
if DATA_DIR.exists():
    for d in DATA_DIR.iterdir():
        if d.is_dir() and (d / 'vector').is_dir():
            data_cities.add(d.name.lower())

metrics_cities = set()
if METRICS_ROOT.exists():
    for d in METRICS_ROOT.iterdir():
        if d.is_dir() and any(d.iterdir()):
            metrics_cities.add(d.name.lower())

all_cities = sorted(data_cities | metrics_cities)
print(f'\nCities in DATA_DIR    : {len(data_cities)}')
print(f'Cities in METRICS_ROOT: {len(metrics_cities)}')
print(f'Total unique cities   : {len(all_cities)}')

rows = []
for city in all_cities:
    cu = city.replace('-', '_')
    data_dir_c    = DATA_DIR     / city
    metrics_dir_c = METRICS_ROOT / city
    vec_dir       = data_dir_c / 'vector'
    aoi_dir       = data_dir_c / 'aoi'
    tiles_path    = data_dir_c / 'tiles' / f'{city}_tiles.gpkg'

    row = {'city': city, 'suitable': suitable_map.get(city)}

    for ds in CAND_NAMES:
        files = list(vec_dir.glob(f'{cu}_{ds}*.parquet')) if vec_dir.exists() else []
        row[f'{ds}_input'] = bool(files)

    for ds in CAND_NAMES:
        row[f'{ds}_output'] = (metrics_dir_c / f'vector_metrics_tiles_{ds}.parquet').exists()

    row['sentinel']     = (metrics_dir_c / SENTINEL).exists()
    row['city_summary'] = (metrics_dir_c / CITY_SUMMARY).exists()
    row['tiles_gpkg']   = tiles_path.exists()

    aoi_files = list(aoi_dir.glob('*.*')) if aoi_dir.exists() else []
    row['has_aoi'] = bool(aoi_files)

    cand_starts = {f'{cu}_{n}' for n in CAND_NAMES}
    ref_files = [
        f for ext in ('*.geojson', '*.gpkg', '*.shp', '*.parquet')
        for f in (vec_dir.glob(ext) if vec_dir.exists() else [])
        if not any(f.stem.lower().startswith(p) for p in cand_starts)
    ]
    row['has_ref'] = bool(ref_files)

    rows.append(row)

audit = pd.DataFrame(rows)
print(f'\nAudit complete: {len(audit)} cities')

In [ ]:
# ── Summary table & flagging ─────────────────────────────────────────────────

has_any_input = audit[[f'{n}_input' for n in CAND_NAMES]].any(axis=1)

# GAP 1: inputs exist but no sentinel
gap1_mask = has_any_input & ~audit['sentinel']
audit['gap1_needs_rerun'] = gap1_mask

can_rerun  = gap1_mask & (audit['suitable'] == True) & audit['has_aoi'] & audit['has_ref']
unsuitable = gap1_mask & (audit['suitable'] != True)
no_aoi     = gap1_mask & (audit['suitable'] == True) & ~audit['has_aoi']
no_ref     = gap1_mask & (audit['suitable'] == True) & audit['has_aoi'] & ~audit['has_ref']

# GAP 2: sentinel exists but tiles GPKG missing (notebook 07 cannot enrich these)
gap2_mask = audit['sentinel'] & ~audit['tiles_gpkg']
audit['gap2_missing_tiles'] = gap2_mask

can_regen_tiles = gap2_mask & audit['has_aoi']
tiles_no_aoi    = gap2_mask & ~audit['has_aoi']

print('=== GAP 1: Unprocessed cities (inputs present, sentinel missing) ===')
print(f'  Total flagged                            : {gap1_mask.sum()}')
print(f'    → suitable + AOI + ref → will re-run  : {can_rerun.sum()}')
print(f'    → suitable=no/unknown (skip)           : {unsuitable.sum()}')
print(f'    → suitable but AOI missing             : {no_aoi.sum()}')
print(f'    → suitable but ref file missing        : {no_ref.sum()}')

if gap1_mask.sum():
    print()
    display_cols = (['city', 'suitable'] + [f'{n}_input' for n in CAND_NAMES]
                    + ['has_aoi', 'has_ref', 'sentinel'])
    print(audit[gap1_mask][display_cols].to_string(index=False))

print()
print('=== GAP 2: Complete cities missing tiles GPKG (notebook 07 cannot enrich) ===')
print(f'  Total missing tiles GPKG                 : {gap2_mask.sum()}')
print(f'    → AOI on disk → can regenerate         : {can_regen_tiles.sum()}')
print(f'    → AOI missing → cannot regenerate      : {tiles_no_aoi.sum()}')

print()
print('=== Already complete ===')
print(f'  Sentinel + tiles GPKG                    : {(audit["sentinel"] & audit["tiles_gpkg"]).sum()}')
print(f'  Sentinel only (tiles cleaned)            : {gap2_mask.sum()}')

audit.to_csv(PROJECT_ROOT / 'outputs' / 'scratch' / 'coverage_audit.csv', index=False)
print(f'\nFull audit saved → outputs/scratch/coverage_audit.csv')

In [ ]:
# ── Re-run pipeline for flagged cities ───────────────────────────────────────
# UrbanValidator.validate_vector() uses overwrite=False internally:
#   - cities with a sentinel are skipped automatically
#   - only cities loaded by load_validation_datasets() (requires AOI on disk) are processed
#
# Cities flagged but lacking AOI will not appear in validator.datasets and
# are reported separately below.

from src.validator import UrbanValidator

rerun_cities = set(audit.loc[can_rerun, 'city'])

if not rerun_cities:
    print('Nothing to re-run — no cities with inputs + AOI + ref but missing sentinel.')
else:
    print(f'Attempting re-run for {len(rerun_cities)} cities: {sorted(rerun_cities)}')
    print('(Cities already complete will be skipped automatically.)')
    print()

    # Write patched config to a temp file (adds root_dir the runner expects)
    _tmp = tempfile.NamedTemporaryFile(suffix='.yaml', delete=False, mode='w')
    yaml.dump(cfg, _tmp)
    _tmp.close()

    v = UrbanValidator(_tmp.name)

    # Filter to only the flagged cities that load_validation_datasets found
    loaded_ids = {ds['id'].lower() for ds in v.datasets}
    to_run = [ds for ds in v.datasets if ds['id'].lower() in rerun_cities]

    not_loaded = rerun_cities - loaded_ids
    if not_loaded:
        print(f'[WARN] {len(not_loaded)} flagged cities not in validator datasets'
              f' (AOI likely missing from disk): {sorted(not_loaded)}')

    print(f'Running {len(to_run)} cities through validate_vector()...')
    results = {}
    for ds in to_run:
        try:
            results[ds['id']] = v._vector_runner.run(ds)
        except Exception as e:
            print(f'[ERR] {ds["id"]}: {e}')
            results[ds['id']] = False

    print()
    print('=== Re-run results ===')
    for city_id, ok in sorted(results.items()):
        status = '[OK]  ' if ok else '[FAIL]'
        print(f'  {status} {city_id}')

    import os; os.unlink(_tmp.name)

In [ ]:
# ── Post-run verification (GAP 1) ─────────────────────────────────────────────
# Re-check sentinel and tiles GPKG for all previously-flagged GAP 1 cities.

print('=== Post-run verification for GAP 1 cities ===')
print(f'{"city":<35} {"sentinel":>10} {"tiles_gpkg":>12}')
print('-' * 60)
newly_complete = 0
still_missing  = []

for city in sorted(audit.loc[audit['gap1_needs_rerun'], 'city']):
    sentinel_ok = (METRICS_ROOT / city / SENTINEL).exists()
    tiles_ok    = (DATA_DIR / city / 'tiles' / f'{city}_tiles.gpkg').exists()
    print(f'  {city:<33} {"✓" if sentinel_ok else "✗":>10} {"✓" if tiles_ok else "✗":>12}')
    if sentinel_ok:
        newly_complete += 1
    else:
        still_missing.append(city)

print()
print(f'Newly complete : {newly_complete}')
print(f'Still missing  : {len(still_missing)}')
if still_missing:
    print(f'  → {still_missing}')
    print('  Likely cause: Suitable=no or AOI file missing from Drive.')

In [ ]:
# ── GAP 2: Regenerate tiles GPKG for complete cities missing the file ─────────
# These are cities notebook 07 skips with "no tiles GPKG".
# The pipeline wrote a sentinel but the tiles file was subsequently cleaned.
# We reconstruct tiles from the AOI using the same make_tiles() call the
# pipeline uses — so tile IDs and geometries are identical.

import geopandas as gpd
from src.utils.tiling import make_tiles
from src.utils.geometry import get_projected_crs
from src.utils.buildings import load_aoi

tile_size_m = float(cfg.get('vector', {}).get('preprocessing', {}).get('tile_size_m', 1000))
cities_to_regen = sorted(audit.loc[can_regen_tiles, 'city'])

if not cities_to_regen:
    print('No cities need tiles GPKG regeneration.')
else:
    print(f'Regenerating tiles GPKG for {len(cities_to_regen)} cities (tile_size={tile_size_m}m)...\n')

    regen_ok, regen_fail, regen_no_aoi = [], [], []

    for city in cities_to_regen:
        aoi_dir    = DATA_DIR / city / 'aoi'
        tiles_path = DATA_DIR / city / 'tiles' / f'{city}_tiles.gpkg'

        # Find any AOI file
        aoi_files = sorted(aoi_dir.glob('*.*')) if aoi_dir.exists() else []
        if not aoi_files:
            print(f'  [SKIP] {city}: no AOI file found')
            regen_no_aoi.append(city)
            continue

        try:
            # Load and dissolve all sub-AOIs (same as pipeline)
            parts = []
            for f in aoi_files:
                try:
                    parts.append(load_aoi(f, crs_out='EPSG:4326'))
                except Exception as e:
                    print(f'  [WARN] {city}: could not load {f.name}: {e}')
            if not parts:
                raise ValueError('all AOI files failed to load')

            aoi = gpd.GeoDataFrame(
                __import__('pandas').concat(parts, ignore_index=True), crs=parts[0].crs
            )
            if len(aoi) > 1:
                aoi = aoi.dissolve().reset_index(drop=True)

            crs  = get_projected_crs(aoi)
            aoi_proj = aoi.to_crs(crs)
            tiles = make_tiles(aoi_proj, tile_size_m)

            tiles_path.parent.mkdir(parents=True, exist_ok=True)
            tiles.to_file(tiles_path, driver='GPKG')
            print(f'  [OK]   {city}: {len(tiles)} tiles → {tiles_path.name}')
            regen_ok.append(city)

        except Exception as e:
            print(f'  [ERR]  {city}: {e}')
            regen_fail.append(city)

    print(f'\nRegenerated OK : {len(regen_ok)}')
    print(f'Failed         : {len(regen_fail)}  {regen_fail}')
    print(f'No AOI on disk : {len(regen_no_aoi)}  {regen_no_aoi}')
    print('\nCities with missing GPKG that still need AOI re-download:')
    print(sorted(audit.loc[tiles_no_aoi, 'city'].tolist()))

In [ ]:
# ── Regenerate vector_all_cities_merged.xlsx ──────────────────────────────────
# Concatenates vector_city_summary_all_datasets.parquet from every processed city.
# Also writes .parquet and .csv for downstream use in notebook 04/06.

summaries = []
missing_summary = []

for city_dir in sorted(METRICS_ROOT.iterdir()):
    if not city_dir.is_dir():
        continue
    p = city_dir / CITY_SUMMARY
    if p.exists():
        try:
            summaries.append(pd.read_parquet(p))
        except Exception as e:
            print(f'[WARN] {city_dir.name}: could not read summary — {e}')
    else:
        missing_summary.append(city_dir.name)

if not summaries:
    print('No city summaries found — nothing to merge.')
else:
    df = pd.concat(summaries, ignore_index=True)

    stem = 'vector_all_cities_merged'
    df.to_parquet(GLOBAL_OUT / f'{stem}.parquet', index=False)
    df.to_csv(    GLOBAL_OUT / f'{stem}.csv',     index=False)

    xlsx_path = GLOBAL_OUT / f'{stem}.xlsx'
    with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name=stem, index=False)

    print(f'Merged {len(summaries)} city summaries → {len(df)} rows, {df["city"].nunique()} cities')
    print(f'  → {xlsx_path}')
    print(f'  → {GLOBAL_OUT / stem}.parquet')
    print(f'  → {GLOBAL_OUT / stem}.csv')

    if missing_summary:
        print(f'\n[WARN] {len(missing_summary)} cities in METRICS_ROOT have no city_summary parquet:')
        print(f'  {missing_summary}')

    print()
    print('=== Column list ===')
    print(df.columns.tolist())
    print()
    print('=== Cities included ===')
    print(sorted(df['city'].unique()))